# LRT article summarization dataset

This notebook collects Lithuanian news articles from LRT and keeps only articles that already include a usable summary. That gives the fine-tuning notebook real article-summary pairs instead of automatically invented labels.


## Imports and settings

The constants below control where the dataset is saved, how many articles are collected, and which LRT sections are searched.


In [1]:
import os
import re
import json
import time
import hashlib
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin

OUTPUT_PATH = "data/lrt_summarization_200.jsonl"
TRAINING_PATH = "data/lrt_summarization_200_training.jsonl"
MAX_ARTICLES = 200

CATEGORY_URLS = [
    "https://www.lrt.lt/naujienos/lietuvoje",
    "https://www.lrt.lt/naujienos/pasaulyje",
    "https://www.lrt.lt/naujienos/verslas",
    "https://www.lrt.lt/naujienos/mokslas-ir-it",
    "https://www.lrt.lt/naujienos/kultura",
    "https://www.lrt.lt/naujienos/sportas",
    "https://www.lrt.lt/naujienos/eismas",
]

HEADERS = {
    "User-Agent": "lt-summarization-research-bot/1.0"
}


## Small text helpers

These helpers keep the article text readable and stable before saving it.


In [2]:
def clean(text):
    if not text:
        return None
    text = text.replace(" ", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def fetch_html(url):
    response = requests.get(url, headers=HEADERS, timeout=25)
    response.raise_for_status()
    response.encoding = "utf-8"
    return response.text


def trim_to_last_sentence(text):
    if not text:
        return None

    text = clean(text)
    if not text:
        return None

    matches = list(re.finditer(r"[.!?\u2026](?=\s|$)", text))
    if not matches:
        return text

    return text[:matches[-1].end()].strip()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def save_jsonl(records, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


## Discover article URLs

Each category page is scanned for LRT news links. Duplicates are removed while preserving the original order.


In [3]:
def discover_urls_from_category(category_url, pages=10):
    urls = []

    for page in range(1, pages + 1):
        page_url = category_url if page == 1 else f"{category_url}?page={page}"

        try:
            html = fetch_html(page_url)
        except Exception as error:
            print(f"Failed category page: {page_url} | {error}")
            continue

        soup = BeautifulSoup(html, "html.parser")

        for link in soup.select("a[href]"):
            href = link.get("href")
            full_url = urljoin("https://www.lrt.lt", href)

            if "/naujienos/" not in full_url:
                continue

            if re.search(r"/naujienos/[^/]+/\d+/\d+/", full_url):
                urls.append(full_url.split("?")[0])

        time.sleep(0.5)

    return list(dict.fromkeys(urls))


## Extract article fields

For each article, the notebook extracts the title, the existing LRT summary, and the main body text.


In [4]:
def extract_title(soup):
    tag = (
        soup.find("meta", property="og:title")
        or soup.find("h1")
        or soup.find("title")
    )

    if not tag:
        return None

    if tag.get("content"):
        return clean(tag.get("content"))

    return clean(tag.get_text(" ", strip=True))


def extract_summary(soup):
    candidates = soup.select(
        "div.article__lead, "
        "div.article-lead, "
        "div.article__content p strong, "
        "article p strong, "
        "div.article__content p, "
        "article p"
    )

    for tag in candidates:
        text = clean(tag.get_text(" ", strip=True))
        if text and len(text.split()) >= 12:
            return trim_to_last_sentence(text)

    meta = (
        soup.find("meta", property="og:description")
        or soup.find("meta", attrs={"name": "description"})
        or soup.find("meta", attrs={"name": "twitter:description"})
    )

    if meta and meta.get("content"):
        text = clean(meta["content"])
        if text and not text.endswith("..."):
            return trim_to_last_sentence(text)

    return None


def remove_summary_from_body(body, summary):
    if not body or not summary:
        return body

    body = body.strip()
    summary = summary.strip()

    if body.startswith(summary):
        return body[len(summary):].strip()

    if body.startswith(summary[:80]):
        match = re.search(r"[.!?\u2026](?=\s|$)", body)
        if match:
            return body[match.end():].strip()

    return body


def extract_body(soup, summary=None):
    blocks = soup.select(
        "div.article__content p, "
        "div.article-content p, "
        "div.article__text p, "
        "article p"
    )

    paragraphs = []

    for paragraph in blocks:
        text = clean(paragraph.get_text(" ", strip=True))

        if not text:
            continue

        if len(text.split()) < 5:
            continue

        lowered = text.lower()

        if lowered.startswith((
            "taip pat skaitykite",
            "skaitykite daugiau",
            "lrt.lt primena",
            "parengė",
            "nuotraukos",
        )):
            continue

        paragraphs.append(text)

    paragraphs = list(dict.fromkeys(paragraphs))

    body = clean(" ".join(paragraphs))
    body = remove_summary_from_body(body, summary)
    body = trim_to_last_sentence(body)

    return body


## Validate and build records

The filters keep examples where the summary is not too short, the article is long enough, and the summary is not too large compared with the article.


In [5]:
def is_valid_record(record):
    summary = record.get("summary")
    text = record.get("text")

    if not summary or not text:
        return False

    summary_words = len(summary.split())
    text_words = len(text.split())

    return (
        12 <= summary_words <= 120
        and 150 <= text_words <= 8_000
        and summary_words / text_words <= 0.45
    )


def extract_record(url):
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    title = extract_title(soup)
    summary = extract_summary(soup)
    text = extract_body(soup, summary)

    record = {
        "url": url,
        "title": title,
        "summary": summary,
        "text": text,
        "summary_words": len(summary.split()) if summary else 0,
        "text_words": len(text.split()) if text else 0,
    }

    if not is_valid_record(record):
        return None

    record["text_hash"] = text_hash(record["text"])
    return record


## Collect the dataset

If a saved dataset already exists, the notebook continues from it. New articles are added until `MAX_ARTICLES` is reached.


In [6]:
records = []

if os.path.exists(OUTPUT_PATH) and os.path.getsize(OUTPUT_PATH) > 0:
    records = pd.read_json(OUTPUT_PATH, lines=True).to_dict("records")
    print(f"Loaded existing records: {len(records)}")

seen_urls = {record["url"] for record in records}
seen_hashes = {record["text_hash"] for record in records if record.get("text_hash")}

urls = []

for category_url in CATEGORY_URLS:
    discovered = discover_urls_from_category(category_url, pages=10)
    print(f"{category_url}: {len(discovered)} URLs")
    urls.extend(discovered)

urls = [url for url in list(dict.fromkeys(urls)) if url not in seen_urls]

print(f"\nTotal new URLs: {len(urls)}")

skipped = []

for url in urls:
    if len(records) >= MAX_ARTICLES:
        break

    try:
        record = extract_record(url)

        if not record:
            skipped.append((url, "invalid"))
            continue

        if record["text_hash"] in seen_hashes:
            skipped.append((url, "duplicate"))
            continue

        records.append(record)
        seen_urls.add(record["url"])
        seen_hashes.add(record["text_hash"])

        print(
            f"[{len(records):>3}/{MAX_ARTICLES}] OK | "
            f"{record['summary_words']} summary | "
            f"{record['text_words']} text | "
            f"{record['title']}"
        )

        if len(records) % 25 == 0:
            save_jsonl(records, OUTPUT_PATH)

        time.sleep(0.7)

    except Exception as error:
        skipped.append((url, str(error)))
        print(f"SKIP | {url} | {error}")

save_jsonl(records, OUTPUT_PATH)

print(f"\nSaved raw records: {len(records)}")
print(f"Skipped: {len(skipped)}")
print(f"Raw output: {OUTPUT_PATH}")


Loaded existing records: 149
https://www.lrt.lt/naujienos/lietuvoje: 22 URLs
https://www.lrt.lt/naujienos/pasaulyje: 29 URLs
https://www.lrt.lt/naujienos/verslas: 37 URLs
https://www.lrt.lt/naujienos/mokslas-ir-it: 33 URLs
https://www.lrt.lt/naujienos/kultura: 35 URLs
https://www.lrt.lt/naujienos/sportas: 30 URLs
https://www.lrt.lt/naujienos/eismas: 26 URLs

Total new URLs: 21

Saved raw records: 149
Skipped: 21
Raw output: data/lrt_summarization_200.jsonl


## Save a training view

The raw file keeps the collection metadata. The training file keeps only the fields needed for summarization fine-tuning.


In [7]:
if records:
    df = pd.DataFrame(records)

    training_df = df.rename(
        columns={
            "text": "input",
            "summary": "target",
        }
    )[["input", "target", "title", "url"]]

    training_df.to_json(
        TRAINING_PATH,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print(f"Training output: {TRAINING_PATH}")
    print(training_df.head(3))
else:
    print("No valid records collected. Increase category pages or loosen filters.")


Training output: data/lrt_summarization_200_training.jsonl
                                               input  \
0  Kalvarijos gimnazijos rūsyje laikinai prisigla...   
1  Visuomenininkas K. Žukauskas, kuris domėjosi „...   
2  „Dienos temoje“ – Lietuvos valstiečių ir žalių...   

                                              target  \
0  Priedangoms modernizuoti savivaldybėse vyriaus...   
1  Visuomenininkui Karoliui Žukauskui ėmus skelbt...   
2  „Nuo pat pradžių mes tikrai skeptiškai buvome ...   

                                               title  \
0  Savivaldybėse atnaujinama civilinė sauga, siūl...   
1  Žukauskas tiria Puchovičiaus vizitus statybose...   
2  Veryga apie Palucką: šio politiko atžvilgiu sk...   

                                                 url  
0  https://www.lrt.lt/naujienos/lietuvoje/2/29190...  
1  https://www.lrt.lt/naujienos/lietuvoje/2/29193...  
2  https://www.lrt.lt/naujienos/lietuvoje/2/29194...  


## Quick sample

This is a simple manual check that the article body and its summary look like a real pair.


In [8]:
sample = pd.DataFrame(records).sample(1, random_state=42).iloc[0]

print("TITLE:", sample["title"])
print("\nSUMMARY:\n", sample["summary"])
print("\nTEXT PREVIEW:\n", sample["text"][:1200])


TITLE: Aistė Adomavičienė. Atsakymas Nerijui Mačiuliui arba Ar tikrai Lietuva rikiuotės gale?

SUMMARY:
 Sakoma, skurdas geriausiai išmoko skaičiuoti. Praėjusią savaitę daug žiniasklaidos dėmesio sulaukė paskelbta žinia apie naują metodiką, pagal kurią Lietuva – skurdžiausia Europos Sąjungos šalis. Pagal daugelį skurdo rodiklių Lietuva jau ne vienerius metus yra tarp prasčiausius rodiklius ES turinčių valstybių. Nors mums ir nesinori šios etiketės savo šaliai, tačiau naujausi Europos statistikos departamento (Eurostat) duomenys patvirtina, kad Lietuvoje skurdo rizikos lygis, kuris 2025 m. šoktelėjo iki 22,6 proc., yra didžiausias ES.

TEXT PREVIEW:
 Savo komentare šią temą palietė ir „Swedbank“ Lietuvoje vyriausiasis ekonomistas Nerijus Mačiulis tvirtinantis, kad skurdo rizikos lygis parodo „santykinį skurdą“ ir pajamų nelygybę, o ne tikrąjį nepriteklių. Ekonominė gerovė nepasiekia pažeidžiamų žmonių Iš dalies sutinkame su ekonomistu, rodiklis nėra tobulas, jam turi įtakos šalies ekono